In [ ]:
import subprocess, anthropic, re

client = anthropic.Anthropic()
prompt = open("prompt_filled.md").read()

for attempt in range(5):
    resp = client.messages.create(
        model="claude-opus-4-7",
        max_tokens=8096,
        messages=[{"role": "user", "content": prompt}]
    )
    text = resp.content[0].text
    code = re.search(r"<code>(.*?)</code>", text, re.DOTALL).group(1)
    open("solution.py", "w").write(code)

    result = subprocess.run(["python", "solution.py"],
                            capture_output=True, text=True, timeout=3600)
    if result.returncode == 0:
        break                        # success
    prompt += f"\n\nTraceback:\n{result.stderr}\nRevise <plan> and <code>."

prompt_filled.md
      │
      ▼
┌─────────────┐     API call      ┌───────────┐
│  Your       │ ────────────────► │  LLM      │
│  Script     │ ◄──────────────── │  (Claude/ │
└─────────────┘   text response   │   GPT)    │
      │                           └───────────┘
      │  parse <code> block
      ▼
 solution.py  ──► python solution.py ──► success? → done
                                         error?   → send traceback back
                                                    (repeat, max 5×)
